# 📈 Backtest Report – Quant Analyst
**Workflow**: Load Signal → Vector Backtest → Extended Metrics → Equity Curve → Monthly PnL Heatmap → Export HTML

---

In [ ]:
from qtrader.analyst import AnalystSession, RoleContext
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

session = AnalystSession(role=RoleContext.ANALYST)
SYMBOL = "BTC-USD"
TIMEFRAME = "1d"
TRANSACTION_COST = 0.0005  # 5 bps

## 1. Prepare Signal

In [ ]:
try:
    df = session.load_ohlcv(SYMBOL, TIMEFRAME)
except Exception:
    df = session.sample_ohlcv(symbol="BTC", days=365)

df = session.make_returns(df)
df = session.add_rolling_features(df, windows=[5, 21])

# Simple momentum signal: go long when close > SMA_21
df = df.with_columns(
    pl.when(pl.col('close') > pl.col('sma_21')).then(1).otherwise(-1).alias('signal')
)
print(f"Signal distribution:\n{df['signal'].value_counts()}")

## 2. Run Vectorized Backtest

In [ ]:
bt = session.run_vector_backtest(df, signal_col='signal', transaction_cost=TRANSACTION_COST)
bt.tail(5)

## 3. Extended Performance Metrics

In [ ]:
metrics = session.compute_extended_metrics(bt['equity_curve'], periods_per_year=252)
# Display as a neat table
metrics_df = pl.DataFrame({'Metric': list(metrics.keys()), 'Value': [round(float(v), 4) if isinstance(v, (int, float)) else v for v in metrics.values()]})
metrics_df

## 4. Equity Curve

In [ ]:
equity = bt['equity_curve'].to_numpy()
bh = (1 + df['returns'].drop_nulls()).cumprod().to_numpy()  # Buy & Hold

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), facecolor='#0f1117',
                                 gridspec_kw={'height_ratios': [3, 1]})
for ax in [ax1, ax2]:
    ax.set_facecolor('#0f1117')
    for sp in ax.spines.values(): sp.set_color('#334155')
    ax.tick_params(colors='#94a3b8')
    ax.grid(color='#1e293b', linestyle='--', linewidth=0.5)

ax1.plot(equity, color='#34d399', linewidth=1.5, label='Strategy')
ax1.plot(bh[:len(equity)], color='#94a3b8', linewidth=1, linestyle='--', label='Buy & Hold')
ax1.set_ylabel('Equity', color='#94a3b8')
ax1.set_title(f'{SYMBOL} Strategy Equity Curve', color='#e2e8f0')
ax1.legend(facecolor='#1e293b', labelcolor='#e2e8f0')

# Drawdown
roll_max = np.maximum.accumulate(equity)
dd = (equity - roll_max) / roll_max
ax2.fill_between(range(len(dd)), dd, alpha=0.7, color='#f87171')
ax2.set_ylabel('Drawdown', color='#94a3b8')
ax2.set_xlabel('Periods', color='#94a3b8')

plt.tight_layout()
plt.show()
equity_fig = fig

## 5. Export HTML Report

In [ ]:
out = session.export_report(
    title=f"Backtest Report – {SYMBOL} Momentum ({TIMEFRAME})",
    sections={
        "Strategy": f"Momentum (Close > SMA_21) | Cost: {TRANSACTION_COST*10000:.1f} bps",
        "Performance Metrics": metrics,
        "Equity Curve & Drawdown": equity_fig,
    },
    path=f"reports/{SYMBOL.replace('-','_')}_Backtest.html",
)
print(f"✅ Report saved: {out}")